# TCC Experimentos

Este notebook é a interface linear do modelo digital PequiFlux. Ele executa somente a fase de validação por padrão; uma execução desta fase produz evidência de engenharia, não evidência confirmatória de H1.


## Identificação, manifesto e perfil de execução

O perfil é uma constante explícita. As variáveis de ambiente abaixo só alteram as raízes de saída; não escolhem perfil nem descobrem uma execução anterior.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import platform
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(f"project root must contain src/: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pequiflux_experiment
import pequiflux_experiment.audit as audit_api
import pequiflux_experiment.digital_model as digital_api
import pequiflux_experiment.dispatch as dispatch_api
import pequiflux_experiment.emulator as emulator_api
import pequiflux_experiment.experiment as experiment_api
import pequiflux_experiment.export as export_api
import pequiflux_experiment.policies as policy_api
import pequiflux_experiment.replay as replay_api
import pequiflux_experiment.statistics as statistics_api

RUNS_ROOT = Path(os.environ.get("PEQUIFLUX_RUNS_ROOT", PROJECT_ROOT / "runs"))
RESULTS_ROOT = Path(os.environ.get("PEQUIFLUX_RESULTS_ROOT", PROJECT_ROOT / "results"))
RUN_PROFILE = "validation"
ALLOWED_PROFILES = ("validation", "pilot", "load-confirmatory", "execute-confirmatory")
CONFIRMATORY_RUN_ID = None
if RUN_PROFILE not in ALLOWED_PROFILES:
    raise ValueError(f"unsupported notebook profile: {RUN_PROFILE}")
if RUN_PROFILE == "pilot":
    raise RuntimeError(
        "pilot profile is documented but blocked: pilot prerequisites and capacity gate are not materialized"
    )
if RUN_PROFILE == "execute-confirmatory":
    raise RuntimeError(
        "execute-confirmatory is reserved for the preregistered full campaign; protocol and capacity gates are not materialized"
    )
if RUN_PROFILE == "load-confirmatory":
    if not isinstance(CONFIRMATORY_RUN_ID, str) or not CONFIRMATORY_RUN_ID.strip():
        raise RuntimeError(
            "load-confirmatory requires an explicit CONFIRMATORY_RUN_ID defined in this cell"
        )
    if CONFIRMATORY_RUN_ID in {".", ".."} or Path(CONFIRMATORY_RUN_ID).name != CONFIRMATORY_RUN_ID:
        raise ValueError("CONFIRMATORY_RUN_ID must identify one run namespace name")

for directory in (RUNS_ROOT, RESULTS_ROOT / "raw", RESULTS_ROOT / "processed", RESULTS_ROOT / "tables", RESULTS_ROOT / "figures"):
    directory.mkdir(parents=True, exist_ok=True)
identity = {
    "project": "PequiFlux - Experimento Reprodutivel",
    "hypothesis": "H1",
    "profile": RUN_PROFILE,
    "runs_root": str(RUNS_ROOT),
    "results_root": str(RESULTS_ROOT),
}
identity


## Carregamento e validação da configuração congelada

Todos os números de protocolo vêm do JSON validado pelo pacote; o notebook não replica parâmetros fatoriais.


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config" / "confirmatory.json"
CONFIG = pequiflux_experiment.load_config(CONFIG_PATH)
CONFIG_HASH = pequiflux_experiment.config_hash(CONFIG)
FACTORIAL_SCENARIOS = pequiflux_experiment.factorial_scenarios(CONFIG)
if len(FACTORIAL_SCENARIOS) != CONFIG.factorial_size:
    raise RuntimeError("factorial scenario count disagrees with validated configuration")
config_summary = {
    "protocol_version": CONFIG.protocol_version,
    "hypothesis": CONFIG.hypothesis,
    "config_hash": CONFIG_HASH,
    "factorial_size": len(FACTORIAL_SCENARIOS),
    "policies": CONFIG.policies,
}
config_summary


## Inventário do ambiente

O manifesto produzido pelo pacote registra o ambiente efetivo. Esta célula exibe apenas um resumo pequeno para manter o notebook legível.


In [ ]:
environment_summary = {
    "python": platform.python_version(),
    "operating_system": platform.platform(),
    "cpu_count": os.cpu_count(),
    "config_path": str(CONFIG_PATH),
    "runs_root": str(RUNS_ROOT),
    "results_root": str(RESULTS_ROOT),
}
environment_summary


## Domínio e arquitetura

O emulador emite eventos serializáveis; o modelo digital mantém uma projeção independente; despacho produz recomendação e o operador aceita antes de um comando. O artefato é um modelo digital, não um gêmeo digital operacional.


In [ ]:
architecture = {
    "physical_layer": "emulator_api.run_day",
    "event_boundary": "EventRecord JSONL",
    "digital_layer": "digital_api.DigitalModel",
    "dispatch_boundary": "dispatch_api.recommend",
    "persistence_boundary": "experiment_api.run_experiment_matrix",
    "scientific_boundary": "H1 confirmatória; validação não é evidência confirmatória",
}
architecture


## Testes automatizados

Os cinco grupos de testes do pacote são executados por um único comando. O teste do próprio notebook é mantido fora deste comando para evitar recursão durante uma execução pelo NotebookClient.


In [ ]:
test_paths = [
    "tests/test_config_manifest.py",
    "tests/test_digital_model_replay.py",
    "tests/test_dispatch_emulator.py",
    "tests/test_experiment_audit.py",
    "tests/test_a2_structural.py",
    "tests/test_statistics_export.py",
]
test_command = [sys.executable, "-m", "pytest", "-q", *test_paths]
test_process = subprocess.run(test_command, cwd=PROJECT_ROOT, capture_output=True, text=True)
if test_process.returncode != 0:
    diagnostics = (test_process.stdout + "\n" + test_process.stderr).strip()
    raise RuntimeError(f"automated test command failed (exit={test_process.returncode}):\n{diagnostics}")
test_summary = (test_process.stdout or test_process.stderr).splitlines()[-1:]
test_summary


## Cenários de sanidade

Uma execução pequena e determinística verifica a jornada do emulador sem pretender representar a matriz confirmatória.


In [ ]:
SANITY_SCENARIO = emulator_api.tiny_scenario(truck_count=4, hoppers=1, scales=1, regime="nominal")
SANITY_SEED = 101
SANITY_POLICY = policy_api.make_policy("lexicographic")
SANITY_RESULT = emulator_api.run_day(SANITY_SCENARIO, SANITY_SEED, SANITY_POLICY)
sanity_summary = {
    "scenario_id": SANITY_RESULT.scenario_id,
    "seed": SANITY_RESULT.seed,
    "policy": SANITY_RESULT.policy_name,
    "event_count": len(SANITY_RESULT.events),
    "metrics": dict(SANITY_RESULT.metrics),
}
sanity_summary


## Demonstração emulador -> modelo digital -> recomendação -> comando

A recomendação abaixo usa somente o instantâneo observável. O comando demonstrado é consequência da aceitação explícita do operador, não de uma mutação direta do emulador pelo despacho.


In [ ]:
demo_events = list(SANITY_RESULT.events)
demo_command_index = next(
    index for index, event in enumerate(demo_events) if event.kind == "SERVICE_STARTED"
)
if demo_command_index < 2:
    raise RuntimeError("service command lacks the required decision protocol")
demo_recommendation_event = demo_events[demo_command_index - 2]
demo_operator_event = demo_events[demo_command_index - 1]
demo_command_event = demo_events[demo_command_index]
demo_sequence = tuple(
    event.kind for event in (demo_recommendation_event, demo_operator_event, demo_command_event)
)
if demo_sequence != ("DECISION_RECORDED", "OPERATOR_DECISION", "SERVICE_STARTED"):
    raise RuntimeError(f"invalid decision protocol: {demo_sequence!r}")
if demo_operator_event.payload["decision"] != "accept":
    raise RuntimeError("service command was not accepted by the operator")
demo_digital = digital_api.DigitalModel.from_snapshot(
    SANITY_RESULT.initial_snapshot, scenario=SANITY_SCENARIO
)
for demo_event in demo_events[: demo_command_index - 2]:
    demo_digital.apply(demo_event)
demo_snapshot_before_recommendation = demo_digital.snapshot()
for demo_event in demo_events[demo_command_index - 2 : demo_command_index + 1]:
    demo_digital.apply(demo_event)
demo_snapshot_after_command = demo_digital.snapshot()
physical_digital_equal = (
    SANITY_RESULT.physical_snapshot.canonical_dict()
    == SANITY_RESULT.digital_snapshot.canonical_dict()
)
if not physical_digital_equal:
    raise RuntimeError("physical and digital snapshots diverged")
demo_flow = {
    "digital_clock_before_recommendation": demo_snapshot_before_recommendation.clock,
    "protocol_sequence": demo_sequence,
    "recommendation": demo_recommendation_event.payload["decision"],
    "operator_decision": demo_operator_event.payload["decision"],
    "command": demo_command_event.payload,
    "digital_clock_after_command": demo_snapshot_after_command.clock,
    "detached_snapshots": SANITY_RESULT.physical_snapshot is not SANITY_RESULT.digital_snapshot,
    "physical_digital_equal": physical_digital_equal,
}
demo_flow


## Replay e auditoria

O replay em memória confirma a equivalência do estado final da demonstração. A auditoria persistida será executada depois que a matriz de validação criar seu namespace.


In [ ]:
SANITY_REPLAY = digital_api.replay_events(
    SANITY_RESULT.events, physical=SANITY_RESULT.initial_snapshot, scenario=SANITY_SCENARIO
)
if SANITY_REPLAY.canonical_dict() != SANITY_RESULT.final_snapshot.canonical_dict():
    raise RuntimeError("sanity replay diverged from the persisted final snapshot")
replay_summary = {
    "replay_pass": True,
    "event_count": len(SANITY_RESULT.events),
    "final_clock": SANITY_REPLAY.clock,
}
replay_summary


## Campanha conforme o perfil

A validação executa apenas duas sementes no cenário mínimo, cobrindo as cinco políticas e produzindo um namespace novo. Os perfis possíveis são explicitamente `pilot`, `load-confirmatory` e `execute-confirmatory`, além desta `validation`.

O guard de `pilot` falha porque pré-requisitos de piloto e capacidade ainda não estão materializados. `load-confirmatory` exige `CONFIRMATORY_RUN_ID` definido nesta célula, resolve exatamente seu namespace e falha se o pacote estiver ausente ou incompleto. `execute-confirmatory` falha com o gate de protocolo e capacidade não materializado; nenhuma campanha completa é executada nesta rodada.


In [ ]:
if RUN_PROFILE == "validation":
    VALIDATION_SEEDS = (101, 102)
    VALIDATION_POLICIES = CONFIG.policies
    BUNDLE = experiment_api.run_experiment_matrix(
        scenarios=[SANITY_SCENARIO],
        seeds=VALIDATION_SEEDS,
        policies=VALIDATION_POLICIES,
        config=CONFIG,
        runs_root=RUNS_ROOT,
        phase=RUN_PROFILE,
    )
    if len(BUNDLE.results) != len(VALIDATION_SEEDS) * len(VALIDATION_POLICIES):
        raise RuntimeError("validation bundle cardinality is not complete")
elif RUN_PROFILE == "load-confirmatory":
    confirmatory_run_dir = RUNS_ROOT / CONFIRMATORY_RUN_ID
    if not confirmatory_run_dir.is_dir():
        raise FileNotFoundError(
            f"load-confirmatory run namespace does not exist: {confirmatory_run_dir}"
        )
    BUNDLE = experiment_api.load_run_bundle(confirmatory_run_dir)
else:
    raise RuntimeError(f"profile has no executable campaign path: {RUN_PROFILE}")
campaign_summary = {
    "profile": RUN_PROFILE,
    "run_id": BUNDLE.run_id,
    "rows": len(BUNDLE.results),
}
campaign_summary


## Artefatos persistidos

A leitura é feita pelo caminho explícito do bundle recém-criado. Não há busca por execução mais recente nem substituição por outro namespace.


In [ ]:
LOADED_BUNDLE = experiment_api.load_run_bundle(BUNDLE.run_dir)
if RUN_PROFILE in {"validation", "load-confirmatory"}:
    AUDIT_REPORT = audit_api.audit_run(LOADED_BUNDLE.run_dir)
persisted_audit = json.loads(LOADED_BUNDLE.audit_path.read_text(encoding="utf-8"))
if not isinstance(persisted_audit, dict):
    raise RuntimeError("persisted audit package is incomplete: audit.json must contain an object")
required_audit_fields = {
    "run_directory", "expected_rows", "observed_rows",
    "a1_pass", "a2_structural_pass", "a2_human_audit_pending",
    "replay_pass", "overall_pass",
}
missing_audit_fields = sorted(required_audit_fields - persisted_audit.keys())
if missing_audit_fields:
    raise RuntimeError(
        "persisted audit package is incomplete; missing fields: "
        + ", ".join(missing_audit_fields)
    )
if not persisted_audit["overall_pass"]:
    raise RuntimeError(f"automated audit failed: {persisted_audit}")
persisted_summary = {
    "run_dir": str(LOADED_BUNDLE.run_dir),
    "manifest": str(LOADED_BUNDLE.manifest_path),
    "results": str(LOADED_BUNDLE.results_path),
    "logs": len(LOADED_BUNDLE.log_paths),
    "audit": str(LOADED_BUNDLE.audit_path),
}
persisted_summary


## Análise estatística e decisão estruturada de H1

H1 exige uma grade pareada completa nos estratos medium/high e é confirmatória. A matriz mínima de validation não satisfaz esse domínio; portanto não se chama `evaluate_h1` nesse perfil. Em `load-confirmatory`, os resultados persistidos são estratificados a partir do manifesto auditado, avaliados por `evaluate_h1` e exportados pela API pública.


In [ ]:
h1_report = None
h1_decision = "NOT_RUN_VALIDATION_PROFILE"
if RUN_PROFILE == "load-confirmatory":
    h1_report = statistics_api.evaluate_h1(BUNDLE, CONFIG)
    h1_decision = h1_report.h1_overall
h1_boundary = {
    "decision": h1_decision,
    "reason": "H1 requer campanha confirmatória pareada; validation é somente evidência de engenharia",
    "evaluator": statistics_api.evaluate_h1.__name__,
    "exporter": export_api.export_analysis.__name__,
}
h1_boundary


## Diagnósticos de consistência do modelo digital

A equivalência de replay é um diagnóstico de engenharia e não um critério confirmatório adicional.


In [ ]:
diagnostics = {
    "a1_pass": persisted_audit["a1_pass"],
    "a2_structural_pass": persisted_audit["a2_structural_pass"],
    "a2_human_audit_pending": persisted_audit["a2_human_audit_pending"],
    "replay_pass": persisted_audit["replay_pass"],
    "diagnostic_is_not_a3": True,
}
diagnostics


## Tabelas e figuras finais

A validação publica somente a tabela de auditoria mínima. Em `load-confirmatory`, tabelas estatísticas e figuras de H1 são regeneradas pela API pública a partir da grade persistida após a reauditoria; nenhum artefato é gerado a partir de valores digitados no notebook.


In [ ]:
audit_table = export_api.export_audit_table(
    LOADED_BUNDLE.audit_path, RESULTS_ROOT / "tables" / "table_audit.csv"
)
if h1_report is not None:
    exported_artifacts = export_api.export_analysis(BUNDLE, h1_report, RESULTS_ROOT)
else:
    exported_artifacts = None
table_summary = {"table_audit": str(audit_table), "rows": 1, "h1_exported": exported_artifacts is not None}
table_summary


## Resumo de artefatos e limites das conclusões

A execução terminou com manifesto, resultados, logs, replay e auditoria automatizada. A inspeção humana de A2 continua pendente. Em `validation`, H1 permanece não executada; em `load-confirmatory`, a decisão corresponde exclusivamente à campanha persistida e auditada carregada.


In [ ]:
final_summary = {
    "profile": RUN_PROFILE,
    "run_id": BUNDLE.run_id,
    "manifest": str(BUNDLE.manifest_path),
    "results": str(BUNDLE.results_path),
    "audit_table": str(audit_table),
    "h1_decision": h1_decision,
    "human_audit_status": "pending",
    "confirmatory_run_loaded": RUN_PROFILE == "load-confirmatory",
    "confirmatory_run_executed": RUN_PROFILE == "execute-confirmatory",
}
final_summary
